
# Emission-line velocity dispersion: narrow [NII] to broad Hα

Narrow-line regions sit at σ_v ≈ 50–300 km/s; a broad Hα component from the
AGN accretion disk reaches thousands of km/s. The [NII] doublet is separated
by 35.4 Å (6549.86 and 6585.28 Å vacuum), which corresponds to σ_v ≈ 1600
km/s — above that the two lines merge into the wing of Hα.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


# ─── Setup ───────────────────────────────────────────────────────────
SIGMA_GRID = np.array([50.0, 200.0, 500.0, 1500.0, 5000.0])
Z = 0.05  # Avoid predict_spectrum NaN at z=0
WAVE_OBS = jnp.linspace(6450.0 * (1 + Z), 6700.0 * (1 + Z), 800)
WAVE_REST = np.asarray(WAVE_OBS) / (1 + Z)

# Load SSP and build model with Cue nebular component (bare-stellar SSP)
ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")
spec = tengri.Spectroscopy(wave_obs=WAVE_OBS, resolution=3000.0)
obs = tengri.Observation(spectroscopy=spec)

model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 3.0,
        "beta": 0.3,
        "tau_gyr": 0.03,
        "log_total_mass": 8.48,
    },
    dust={
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_bc": 0.0,
        "tau_diff": 0.05,
    },
    neb={
        "type": "cue",
        "all_params": tengri.FIXED,
        "logU": tengri.Fixed(-2.5),
        "fesc": tengri.Fixed(0.0),
    },
    redshift=tengri.Fixed(Z),
)

baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

# ─── Sweep emission-line sigma_v ──────────────────────────────────
norm = mpl.colors.Normalize(vmin=SIGMA_GRID.min(), vmax=SIGMA_GRID.max())
cmap = plt.get_cmap("viridis")

fig, ax = plt.subplots(figsize=(7.0, 4.6))
for sigma in SIGMA_GRID:
    params = {**baseline, "sigma_v_kms": jnp.float64(sigma)}
    spec_out = model.predict_spectrum(params, wave_obs=WAVE_OBS)
    flux = np.asarray(spec_out)

    # Continuum normalization at 6600 Å
    cont_mask = (WAVE_REST >= 6600) & (WAVE_REST <= 6650)
    f_cont = np.median(flux[cont_mask])
    ax.plot(WAVE_REST, flux / f_cont, color=cmap(norm(sigma)), lw=1.2)

# Mark emission lines
ax.axvline(6549.86, color="0.55", lw=0.4, ls=":")  # [NII] 6548
ax.axvline(6564.61, color="0.55", lw=0.4, ls=":")  # Hα 6565
ax.axvline(6585.28, color="0.55", lw=0.4, ls=":")  # [NII] 6584
ax.text(6564.61, 1.20, "Hα", fontsize=8, color="0.4", ha="center")
ax.text(6549.86, 1.27, "[NII]", fontsize=7, color="0.4", ha="center")

ax.set(
    xlim=(6480, 6680),
    yscale="log",
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$F_\lambda\,/\,F_{\rm cont}$ (normalized at 6600-6650 Å)",
)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.01)
cb.set_label(r"$\sigma_v$  [km s$^{-1}$]")

fig.tight_layout()
plt.savefig("plot_velocity_offset_lines.png", dpi=150, bbox_inches="tight")